# MDLF Training : rendu du bonhomme (GPU)
Exécution > Tout exécuter, puis coller le jeton Hugging Face quand il est demandé. Durée : environ 15-25 min.

In [ ]:
# MDLF perso : rendu GPU en ligne (Colab ou Kaggle). Une seule cellule : tout exécuter.
# Colab : Exécution > Modifier le type d'exécution > GPU T4, puis « Tout exécuter » et coller le jeton HF demandé.
# Le jeton est saisi (getpass) et n'est jamais écrit dans le carnet. Kaggle : secret HF_TOKEN (Add-ons > Secrets).
import json, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient; TOK = UserSecretsClient().get_secret("HF_TOKEN"); OU = "kaggle"
except Exception:
    from getpass import getpass
    TOK = os.environ.get("HF_TOKEN") or getpass("Jeton Hugging Face (hf_...) puis Entrée : ").strip()
    OU = "colab" if "google.colab" in sys.modules or os.path.exists("/content") else "autre"
os.environ["HF_TOKEN"] = TOK
sh("nvidia-smi -L || echo 'PAS DE GPU : Exécution > Modifier le type d exécution > T4 GPU, puis relancer'")
sh("pip -q install -U huggingface_hub")
from huggingface_hub import HfApi, hf_hub_download
REPO = "Maxencedlf/mdlf-perso-rendu"
API = HfApi(token=TOK)
def etat(msg):   # suivi à distance : etat/<tag>.txt sur le dépôt (Claude le lit sans voir l'écran)
    global JOURNAL
    JOURNAL = globals().get("JOURNAL", "") + f"{time.strftime('%H:%M:%S')} {msg}\n"; print(msg)
    try:
        API.upload_file(path_or_fileobj=JOURNAL.encode(), path_in_repo=f"etat/{globals().get('TAG', 'debut')}.txt",
                        repo_id=REPO, repo_type="dataset")
    except Exception as e:
        print("etat non envoyé", e)
import traceback
try:
    TAG = open(hf_hub_download(REPO, "paquet/DERNIER.txt", repo_type="dataset", token=TOK, force_download=True)).read().strip()
except Exception:
    TAG = "inconnu"
etat("démarré sur " + subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip())
W = "/tmp/mdlf"; os.makedirs(W, exist_ok=True); os.chdir(W)
BL = "/tmp/blender-5.2.3-linux-x64"
if not os.path.exists(BL + "/blender"):
    sh("apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null || true")
    sh("curl -sSLo /tmp/b.tar.xz https://download.blender.org/release/Blender5.2/blender-5.2.3-linux-x64.tar.xz")
    sh("echo '78486c018b4443efe5e75ab54c290ed8c15abafd249b862c2c9ce2530a918eec  /tmp/b.tar.xz' | sha256sum -c")
    sh("tar -xf /tmp/b.tar.xz -C /tmp")
etat("Blender prêt")
job = json.load(open(hf_hub_download(REPO, f"paquet/{TAG}.json", repo_type="dataset", token=TOK, force_download=True)))
with tarfile.open(hf_hub_download(REPO, f"paquet/{TAG}.tar.gz", repo_type="dataset", token=TOK, force_download=True)) as tf:
    tf.extractall(W, filter="data")
# les cheveux pointent vers le chemin absolu du PC : même chemin ici
os.makedirs(os.path.dirname(job["assets_abs"]), exist_ok=True)
if not os.path.exists(job["assets_abs"]):
    os.symlink(W + "/mh-assets/sys", job["assets_abs"])
OUT = W + "/out/r"; os.makedirs(OUT + "/raw", exist_ok=True)
for f in os.listdir(W + "/raw"):
    shutil.copy(W + "/raw/" + f, OUT + "/raw/" + f)
env = dict(os.environ, BLENDER=BL + "/blender", MDLF_GPU="1")
cmd = [OUT if c == "OUT" else c for c in job["cmd"]]
t1 = time.time()
etat("rendu lancé : " + " ".join(cmd))
r = subprocess.run(cmd, cwd=W, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-3000:])
etat(f"rendu fini, code {r.returncode}" + ("" if r.returncode == 0 else "\n" + r.stdout[-2000:]))
dt = time.time() - t1
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
json.dump({"tag": TAG, "ou": OU, "gpu": gpu, "rendu_s": round(dt), "total_s": round(time.time() - T0), "code": r.returncode},
          open(OUT + "/raw/temps_en_ligne.json", "w"), indent=1)
with tarfile.open(W + f"/{TAG}.tar.gz", "w:gz") as tf:
    for f in os.listdir(OUT + "/raw"):
        if f.endswith((".png", ".json")):
            tf.add(OUT + "/raw/" + f, arcname=f)
HfApi(token=TOK).upload_file(path_or_fileobj=W + f"/{TAG}.tar.gz", path_in_repo=f"resultats/{TAG}.tar.gz",
                             repo_id=REPO, repo_type="dataset")
etat("FINI, résultats envoyés")
print("FINI", TAG, f"rendu {dt / 60:.1f} min", gpu, "code", r.returncode)
